# Sprint 4 — Data Science and Statistical Computing
**Transcrição → Processamento → Modelo → Indicadores → Recomendação**

Este notebook roda de ponta a ponta no Google Colab (Ambiente de execução > Executar tudo).
Atenção: os dados de treino são **sintéticos** (o repositório não tinha as transcrições reais);
para usar dados reais, substitua `data/transcricoes_treino.csv` (colunas `id, transcricao, produto, sentimento, churn, oportunidade`).

## 1. Preparação (tudo autônomo: o código-fonte vai dentro do próprio notebook)
As células abaixo gravam os módulos em `src/`, instalam as dependências e geram os dados de exemplo.
Não precisa de GitHub nem de upload de arquivos.

In [ ]:
import os
os.makedirs("src", exist_ok=True); os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True); os.makedirs("reports", exist_ok=True)
!pip -q install scikit-learn pandas numpy joblib

In [ ]:
%%writefile src/__init__.py

In [ ]:
%%writefile src/config.py
"""Configuração central do projeto."""
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"

TREINO_CSV = DATA_DIR / "transcricoes_treino.csv"
NOVAS_SINTETICAS_CSV = DATA_DIR / "novas_sinteticas.csv"
NOVAS_MANUAIS_CSV = DATA_DIR / "novas_manuais.csv"
NOVAS_CEGAS_CSV = DATA_DIR / "novas_cegas.csv"
MODELO_PATH = MODELS_DIR / "modelos.joblib"

SEED = 42

PRODUTOS = ["Conta Digital", "Cartão de Crédito", "Empréstimo", "Seguro", "Investimentos"]
SENTIMENTOS = ["negativo", "neutro", "positivo"]

# alvos supervisionados -> coluna do CSV
TAREFAS = {
    "produto": "produto",
    "sentimento": "sentimento",
    "churn": "churn",
    "oportunidade": "oportunidade",
}

In [ ]:
%%writefile src/processing.py
"""Etapa de Processamento: limpeza, normalização e extração de termos."""
import re
import unicodedata
from collections import Counter

STOPWORDS = set("""
a o as os um uma uns umas de do da dos das em no na nos nas por para com sobre ao aos
e ou mas que se como mais menos muito muita muitos muitas ja sim eu voce vocês voces me te
meu minha meus minhas seu sua seus suas ele ela eles elas nos lhe isso isto esse essa esses essas
este esta estes estas aquele aquela ser estar estou esta estamos sao foi era tem tenho ter tinha
vou vai vamos pode posso podem poder quero queria gostaria preciso favor bom dia boa tarde noite
ola oi tudo bem obrigado obrigada ai la aqui so apenas tambem entao pois porque quando onde qual
quais algum alguma coisa vez vezes hoje agora momento instante claro certo perfeito atendente
cliente ligando contato ajudar ajuda pra pro numa num dele dela deles delas ate sido ainda
mesmo mesma depois antes durante essa isso fazer faz feito
""".split())


def remover_acentos(txt: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", txt) if unicodedata.category(c) != "Mn")


def remover_falantes(txt: str) -> str:
    """Remove os rótulos 'Atendente:' / 'Cliente:' do início das linhas."""
    return re.sub(r"(?im)^\s*(atendente|cliente|agente|consultor)\s*:\s*", "", txt)


def limpar(txt: str) -> str:
    """Minúsculas, sem rótulos de falante, sem acentos, sem pontuação/dígitos soltos."""
    txt = remover_falantes(str(txt)).lower()
    txt = remover_acentos(txt)
    txt = re.sub(r"https?://\S+|\S+@\S+", " ", txt)
    txt = re.sub(r"\d+", " ", txt)
    txt = re.sub(r"[^a-z\s]", " ", txt)
    return re.sub(r"\s+", " ", txt).strip()


def tokens(txt: str) -> list[str]:
    return [t for t in limpar(txt).split() if t not in STOPWORDS and len(t) > 2]


def texto_processado(txt: str) -> str:
    """Texto limpo, sem stopwords, usado como entrada dos modelos (TF-IDF)."""
    return " ".join(tokens(txt))


def termos_frequentes(txt: str, n: int = 8) -> list[tuple[str, int]]:
    return Counter(tokens(txt)).most_common(n)

In [ ]:
%%writefile src/data_gen.py
"""Gerador de transcrições sintéticas (pt-BR) de atendimento comercial.

Serve como substituto dos dados reais enquanto eles não estão no repositório.
Para usar dados reais, basta gerar um CSV com as colunas:
    id, transcricao, produto, sentimento, churn, oportunidade
"""
import argparse
import random

import pandas as pd

from . import config

SAUD = [
    "Atendente: Bom dia, em que posso ajudar?",
    "Atendente: Olá, obrigado por entrar em contato. Como posso ajudar hoje?",
    "Atendente: Boa tarde! Aqui é da central de relacionamento, qual o motivo do contato?",
    "Atendente: Oi, tudo bem? Pode me explicar o que aconteceu?",
]

PRODUTO_FRASES = {
    "Conta Digital": [
        "Cliente: Estou ligando sobre a minha conta digital.",
        "Cliente: Tenho uma dúvida sobre o aplicativo da conta corrente, o pix e as transferências.",
        "Cliente: É sobre a conta digital, o saldo e o extrato do app.",
        "Cliente: Preciso falar do pix e da tarifa da minha conta.",
    ],
    "Cartão de Crédito": [
        "Cliente: Estou ligando por causa do meu cartão de crédito.",
        "Cliente: É sobre a fatura do cartão e o limite.",
        "Cliente: Tenho dúvida na anuidade do cartão e nas parcelas da fatura.",
        "Cliente: Meu cartão, a bandeira e o limite de crédito.",
    ],
    "Empréstimo": [
        "Cliente: Estou ligando sobre o meu empréstimo pessoal.",
        "Cliente: É sobre as parcelas do financiamento e a taxa de juros.",
        "Cliente: Quero falar do crédito consignado, o contrato e o saldo devedor.",
        "Cliente: Tenho dúvida na simulação do empréstimo e na quitação.",
    ],
    "Seguro": [
        "Cliente: Estou ligando sobre o meu seguro.",
        "Cliente: É sobre a apólice e a cobertura do seguro de vida.",
        "Cliente: Preciso falar do sinistro e do prêmio do seguro do carro.",
        "Cliente: Tenho dúvida na renovação da apólice e na franquia.",
    ],
    "Investimentos": [
        "Cliente: Estou ligando sobre meus investimentos.",
        "Cliente: É sobre o CDB, a rentabilidade e o resgate.",
        "Cliente: Quero falar da carteira de investimentos, tesouro direto e fundos.",
        "Cliente: Tenho dúvida na corretora e no rendimento da aplicação.",
    ],
}

CORPO = {
    "negativo": [
        "Cliente: Estou muito insatisfeito, já é a terceira vez que ligo e ninguém resolve.",
        "Cliente: Isso é um absurdo, o atendimento é péssimo e demora demais.",
        "Cliente: Fui cobrado indevidamente e ninguém me dá retorno, estou bastante irritado.",
        "Cliente: O sistema vive travando, o aplicativo não funciona e já perdi muito tempo.",
        "Cliente: Estou decepcionado, prometeram resolver e não cumpriram nada.",
        "Cliente: Que falta de respeito, reclamei no protocolo e continua o mesmo problema.",
        "Cliente: As taxas são abusivas e ninguém explica nada, estou revoltado.",
    ],
    "neutro": [
        "Cliente: Queria apenas entender como funciona e confirmar uma informação.",
        "Cliente: Preciso de uma informação sobre prazos e documentos.",
        "Cliente: Gostaria de atualizar meu cadastro e saber os horários de atendimento.",
        "Cliente: Só estou verificando uma data e o valor atual.",
        "Cliente: Pode me enviar o comprovante por e-mail, por favor?",
        "Cliente: Quero saber qual é o procedimento para solicitar a segunda via.",
    ],
    "positivo": [
        "Cliente: Estou muito satisfeito, o atendimento sempre foi excelente.",
        "Cliente: Adoro o serviço, é prático e funciona muito bem, estou contente.",
        "Cliente: Gostei muito da solução que vocês deram, foi rápido e eficiente.",
        "Cliente: Parabéns pelo atendimento, resolveram tudo de forma atenciosa.",
        "Cliente: Recomendo para os amigos, a experiência tem sido ótima.",
        "Cliente: Fiquei feliz com o resultado, muito obrigado pela ajuda.",
    ],
}

# frases "ruído": sentimento contrário/ambíguo para não deixar a tarefa trivial
RUIDO = {
    "negativo": ["Cliente: Apesar de gostar da marca, o problema persiste.",
                 "Cliente: Antes era bom, mas agora está complicado."],
    "neutro": ["Cliente: Não tenho reclamação, mas também não tenho elogio.",
               "Cliente: Está tudo normal, só preciso conferir."],
    "positivo": ["Cliente: Teve uma demora no começo, mas depois resolveram.",
                 "Cliente: Tive um pequeno problema, mas foi superado."],
}

CHURN = [
    "Cliente: Se não resolver hoje vou cancelar tudo.",
    "Cliente: Estou pensando em encerrar a conta e mudar para outro banco.",
    "Cliente: Já fui na concorrência e a proposta deles é melhor, vou migrar.",
    "Cliente: Quero cancelar o serviço e pedir a portabilidade para outra instituição.",
    "Cliente: Vou fechar tudo e procurar outra empresa que respeite o cliente.",
    "Cliente: Cansei, quero o cancelamento imediato e não pretendo continuar.",
]

CHURN_AMBIGUO = [  # fala parecida mas sem intenção real de sair (ruído)
    "Cliente: Não quero cancelar, só quero resolver isso.",
    "Cliente: Já pensei em sair, mas prefiro continuar com vocês.",
]

OPORT = {
    "Conta Digital": ["Cliente: Tenho interesse em conhecer o cartão e um limite maior de pix.",
                      "Cliente: Queria saber se há pacote de serviços com mais benefícios."],
    "Cartão de Crédito": ["Cliente: Gostaria de aumentar o limite e conhecer o cartão black com mais benefícios.",
                          "Cliente: Tenho interesse em um programa de pontos e cashback melhor."],
    "Empréstimo": ["Cliente: Tenho interesse em contratar um novo crédito com taxa menor.",
                   "Cliente: Queria simular um financiamento do imóvel, pode me enviar a proposta?"],
    "Seguro": ["Cliente: Gostaria de contratar também um seguro residencial, pode fazer uma cotação?",
               "Cliente: Tenho interesse em ampliar a cobertura e incluir minha família."],
    "Investimentos": ["Cliente: Quero investir mais, o que vocês têm de produto com melhor rendimento?",
                      "Cliente: Tenho interesse em abrir uma previdência e diversificar a carteira."],
}

FECHO = {
    "negativo": ["Atendente: Entendo, vou escalar o seu caso com prioridade.",
                 "Atendente: Peço desculpas pelo transtorno, vou abrir um protocolo urgente."],
    "neutro": ["Atendente: Certo, a informação foi registrada.",
               "Atendente: Perfeito, enviei os detalhes para o seu e-mail."],
    "positivo": ["Atendente: Fico feliz em ajudar, qualquer coisa estamos à disposição!",
                 "Atendente: Obrigado pelo feedback, vou registrar o elogio."],
}

FILLER = [
    "Atendente: Pode confirmar seu CPF, por favor?",
    "Atendente: Um momento enquanto verifico no sistema.",
    "Cliente: Claro, um instante.",
    "Atendente: Já localizei o seu cadastro.",
]


def gerar_uma(rng: random.Random, i: int) -> dict:
    produto = rng.choice(config.PRODUTOS)
    sent = rng.choices(config.SENTIMENTOS, weights=[0.35, 0.30, 0.35])[0]
    p_churn = {"negativo": 0.50, "neutro": 0.08, "positivo": 0.02}[sent]
    p_oport = {"negativo": 0.05, "neutro": 0.30, "positivo": 0.45}[sent]
    churn = int(rng.random() < p_churn)
    oport = int(rng.random() < p_oport)

    linhas = [rng.choice(SAUD), rng.choice(PRODUTO_FRASES[produto])]
    if rng.random() < 0.12:  # menciona um segundo produto de passagem
        outro = rng.choice([p for p in config.PRODUTOS if p != produto])
        linhas.append(rng.choice(PRODUTO_FRASES[outro]).replace("Estou ligando", "Também tenho"))
    linhas += rng.sample(CORPO[sent], k=rng.randint(1, 2))
    if rng.random() < 0.25:
        linhas.append(rng.choice(RUIDO[sent]))
    if rng.random() < 0.5:
        linhas.append(rng.choice(FILLER))
    if churn:
        linhas.append(rng.choice(CHURN))
    elif rng.random() < 0.06:
        linhas.append(rng.choice(CHURN_AMBIGUO))
    if oport:
        linhas.append(rng.choice(OPORT[produto]))
    linhas.append(rng.choice(FECHO[sent]))

    # ruído de rótulo: 4% de erro de anotação no sentimento, simulando anotação humana
    sent_rot = sent
    if rng.random() < 0.04:
        sent_rot = rng.choice([s for s in config.SENTIMENTOS if s != sent])
    return {"id": i, "transcricao": "\n".join(linhas), "produto": produto,
            "sentimento": sent_rot, "churn": churn, "oportunidade": oport}


def gerar(n: int, seed: int) -> pd.DataFrame:
    rng = random.Random(seed)
    return pd.DataFrame([gerar_uma(rng, i) for i in range(n)])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--treino", type=int, default=800)
    ap.add_argument("--novas", type=int, default=150)
    a = ap.parse_args()
    config.DATA_DIR.mkdir(exist_ok=True)
    gerar(a.treino, config.SEED).to_csv(config.TREINO_CSV, index=False)
    gerar(a.novas, config.SEED + 1000).to_csv(config.NOVAS_SINTETICAS_CSV, index=False)
    print(f"Gerados {a.treino} (treino) e {a.novas} (novas) em {config.DATA_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/manuais_cegas.py
"""Lote CEGO de transcrições: escrito depois de calibrar regras e modelo e NUNCA usado para ajustes.

Rótulos: (transcricao, produto, sentimento, churn, oportunidade).
"""

CASOS_CEGOS = [
    ("Atendente: Bom dia, pois não?\nCliente: Oi, o meu investimento em fundos caiu bastante este mês e ninguém me avisou nada. Estou muito preocupado e bravo com isso.\nAtendente: Entendo, vou explicar a oscilação.\nCliente: Se continuar assim vou tirar meu dinheiro e levar para outra corretora.",
     "Investimentos", "negativo", 1, 0),
    ("Atendente: Central, boa tarde.\nCliente: Boa tarde! Só liguei para elogiar a moça que me atendeu semana passada no empréstimo, foi atenciosa demais. Estou tranquilo com as parcelas.\nAtendente: Que bom, vou repassar.",
     "Empréstimo", "positivo", 0, 0),
    ("Atendente: Olá, como ajudo?\nCliente: Preciso da segunda via do boleto da apólice e saber até quando posso pagar sem multa.\nAtendente: Até o dia dez.\nCliente: Beleza, valeu.",
     "Seguro", "neutro", 0, 0),
    ("Atendente: Bom dia!\nCliente: Bom dia, vi no site que vocês têm cartão com cashback. Tenho interesse, como faço para trocar o meu atual?\nAtendente: Posso fazer a solicitação agora.\nCliente: Sim, por favor, gostei da ideia.",
     "Cartão de Crédito", "positivo", 0, 1),
    ("Atendente: Em que posso ajudar?\nCliente: A tarifa mensal da conta subiu e eu não fui comunicado. Isso é falta de respeito com o cliente. Vou pedir o encerramento da conta e abrir em outro banco.\nAtendente: Posso isentar a tarifa.\nCliente: Vou pensar.",
     "Conta Digital", "negativo", 1, 0),
    ("Atendente: Boa noite.\nCliente: Queria confirmar se meu pix de ontem foi compensado. Só isso.\nAtendente: Sim, foi compensado.\nCliente: Obrigado.",
     "Conta Digital", "neutro", 0, 0),
    ("Atendente: Olá!\nCliente: Oi, estou com dificuldade para entender o simulador de financiamento do imóvel, mas gostaria de contratar o crédito se a taxa for boa. Pode me mandar uma proposta?\nAtendente: Envio agora.",
     "Empréstimo", "neutro", 0, 1),
    ("Atendente: Bom dia.\nCliente: Meu seguro de vida cobriu tudo direitinho, fiquei muito satisfeito. Não penso em trocar de seguradora, sou cliente há dez anos.\nAtendente: Agradecemos a confiança.",
     "Seguro", "positivo", 0, 0),
]

In [ ]:
%%writefile src/manuais.py
"""Transcrições escritas à mão, com vocabulário diferente do gerador sintético.

São o teste mais honesto do pipeline: não vieram do mesmo processo que gerou o treino.
Gera data/novas_manuais.csv.  Rótulos: produto, sentimento, churn (0/1), oportunidade (0/1).
"""
import pandas as pd

from . import config

CASOS = [
    ("Atendente: Central de atendimento, bom dia.\nCliente: Bom dia. Meu cartão foi cobrado duas vezes na fatura e já liguei ontem sem solução. Estou bem chateado.\nAtendente: Vou abrir uma contestação.\nCliente: Se isso não for estornado essa semana, eu cancelo o cartão e vou para a concorrência.",
     "Cartão de Crédito", "negativo", 1, 0),
    ("Atendente: Olá, como posso ajudar?\nCliente: Oi! Queria agradecer, o pix caiu na hora e o app está ótimo. Aliás, vocês têm algum pacote com mais benefícios para quem movimenta bastante?\nAtendente: Temos sim, posso enviar a proposta.\nCliente: Pode mandar, tenho interesse.",
     "Conta Digital", "positivo", 0, 1),
    ("Atendente: Boa tarde.\nCliente: Preciso saber a data de vencimento da próxima parcela do meu financiamento e o saldo devedor atual.\nAtendente: Um momento, por favor.\nCliente: Obrigado, anotei aqui.",
     "Empréstimo", "neutro", 0, 0),
    ("Atendente: Em que posso ajudar?\nCliente: Meu seguro do carro venceu e a renovação veio quarenta por cento mais cara. Um absurdo, sempre paguei em dia.\nAtendente: Posso revisar a franquia.\nCliente: Já cotei em outra seguradora e é bem mais barato. Vou sair se não baixarem o prêmio.",
     "Seguro", "negativo", 1, 0),
    ("Atendente: Bom dia!\nCliente: Bom dia, tenho um dinheiro parado e quero investir mais. O CDB que tenho rende pouco, o que vocês recomendam com melhor rentabilidade?\nAtendente: Posso apresentar o tesouro direto e fundos.\nCliente: Perfeito, adorei, me envia por e-mail.",
     "Investimentos", "positivo", 0, 1),
    ("Atendente: Atendimento, boa noite.\nCliente: O aplicativo da minha conta não abre desde ontem. Preciso pagar um boleto e estou irritado, isso é a terceira vez este mês.\nAtendente: Peço desculpas.\nCliente: Não quero cancelar a conta, mas assim fica difícil, resolvam isso.",
     "Conta Digital", "negativo", 0, 0),
    ("Atendente: Olá!\nCliente: Oi, gostaria de aumentar o limite do meu cartão, sempre pago a fatura em dia e estou muito satisfeito com o serviço.\nAtendente: Vou verificar a análise de crédito.\nCliente: Ótimo, obrigado!",
     "Cartão de Crédito", "positivo", 0, 1),
    ("Atendente: Bom dia, central de crédito.\nCliente: Fiz a simulação do empréstimo pessoal e as taxas de juros estão abusivas. Pedi portabilidade para outro banco que oferece metade da taxa.\nAtendente: Posso tentar cobrir a oferta.\nCliente: Já decidi, quero encerrar o contrato e quitar tudo hoje.",
     "Empréstimo", "negativo", 1, 0),
    ("Atendente: Boa tarde, em que posso ajudar?\nCliente: Queria entender como funciona o resgate da minha aplicação e se há alguma taxa. Só isso mesmo.\nAtendente: O resgate é no mesmo dia útil.\nCliente: Certo, entendi.",
     "Investimentos", "neutro", 0, 0),
    ("Atendente: Olá, tudo bem?\nCliente: Tudo ótimo! Acionei o sinistro do meu seguro residencial semana passada e foi resolvido super rápido, parabéns. Posso incluir meu carro na mesma apólice? Quero fazer uma cotação.\nAtendente: Claro, envio ainda hoje.",
     "Seguro", "positivo", 0, 1),
    ("Atendente: Bom dia.\nCliente: Preciso atualizar meu endereço no cadastro da conta e baixar o extrato dos últimos três meses.\nAtendente: Posso enviar por e-mail.\nCliente: Pode ser, obrigado.",
     "Conta Digital", "neutro", 0, 0),
    ("Atendente: Em que posso ajudar?\nCliente: Estou decepcionado com a anuidade do cartão, ninguém me avisou da cobrança. Quero falar com o gerente, senão encerro tudo e abro conta em outro lugar.\nAtendente: Vou transferir.",
     "Cartão de Crédito", "negativo", 1, 0),
]

from .manuais_cegas import CASOS_CEGOS  # noqa: E402

def main():
    df = pd.DataFrame(CASOS, columns=["transcricao", "produto", "sentimento", "churn", "oportunidade"])
    df.insert(0, "id", range(len(df)))
    df.to_csv(config.NOVAS_MANUAIS_CSV, index=False)
    cg = pd.DataFrame(CASOS_CEGOS, columns=df.columns[1:])
    cg.insert(0, "id", range(len(cg)))
    cg.to_csv(config.NOVAS_CEGAS_CSV, index=False)
    print(f"{len(df)} manuais + {len(cg)} cegas gravadas em {config.DATA_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/train.py
"""Etapa de Modelo: compara candidatos por tarefa, seleciona o melhor e salva.

Uso:  python -m src.train
Saídas: models/modelos.joblib, reports/comparacao_modelos.csv, reports/modelo_selecionado.md
"""
import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from . import config
from .processing import texto_processado

# ordem = preferência em caso de empate (do mais simples/interpretável ao mais complexo)
CANDIDATOS = {
    "LogisticRegression": lambda: LogisticRegression(C=5, max_iter=2000, class_weight="balanced"),
    "NaiveBayes": lambda: MultinomialNB(alpha=0.3),
    "LinearSVC (calibrado)": lambda: CalibratedClassifierCV(LinearSVC(C=1.0, class_weight="balanced"), cv=3),
    "RandomForest": lambda: RandomForestClassifier(n_estimators=300, random_state=config.SEED,
                                                   class_weight="balanced", n_jobs=-1),
}


def montar(nome: str) -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("clf", CANDIDATOS[nome]()),
    ])


def carregar_treino() -> pd.DataFrame:
    if not config.TREINO_CSV.exists():
        raise SystemExit("Dados de treino não encontrados. Rode antes: python -m src.data_gen")
    df = pd.read_csv(config.TREINO_CSV)
    df["texto"] = df["transcricao"].map(texto_processado)
    return df


def main():
    df = carregar_treino()
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=config.SEED)
    linhas, escolhidos = [], {}

    for tarefa, col in config.TAREFAS.items():
        y = df[col]
        X_tr, X_te, y_tr, y_te = train_test_split(
            df["texto"], y, test_size=0.2, stratify=y, random_state=config.SEED)
        melhor = None
        for nome in CANDIDATOS:
            f1_cv = cross_val_score(montar(nome), X_tr, y_tr, cv=cv, scoring="f1_macro", n_jobs=1)
            m = montar(nome).fit(X_tr, y_tr)
            pred = m.predict(X_te)
            linha = {"tarefa": tarefa, "modelo": nome,
                     "f1_macro_cv": round(f1_cv.mean(), 4), "desvio_cv": round(f1_cv.std(), 4),
                     "f1_macro_holdout": round(f1_score(y_te, pred, average="macro"), 4),
                     "acuracia_holdout": round(accuracy_score(y_te, pred), 4)}
            linhas.append(linha)
            # seleção: maior F1 CV; empate (diferença < 0,005) -> modelo mais simples (vem antes)
            if melhor is None or linha["f1_macro_cv"] > melhor["f1_macro_cv"] + 0.005:
                melhor = linha
        escolhidos[tarefa] = melhor
        print(f"[{tarefa}] melhor: {melhor['modelo']} (F1 CV={melhor['f1_macro_cv']})")

    # reajusta os escolhidos com todos os dados de treino
    modelos = {t: montar(e["modelo"]).fit(df["texto"], df[config.TAREFAS[t]])
               for t, e in escolhidos.items()}

    config.MODELS_DIR.mkdir(exist_ok=True)
    config.REPORTS_DIR.mkdir(exist_ok=True)
    joblib.dump({"modelos": modelos, "escolhidos": escolhidos}, config.MODELO_PATH)
    comp = pd.DataFrame(linhas)
    comp.to_csv(config.REPORTS_DIR / "comparacao_modelos.csv", index=False)
    escrever_justificativa(comp, escolhidos)
    print("Modelos salvos em", config.MODELO_PATH)


def escrever_justificativa(comp: pd.DataFrame, escolhidos: dict):
    md = ["# Modelo final selecionado e justificativa", "",
          "Para cada tarefa (produto, sentimento, risco de churn, oportunidade comercial) foram "
          "comparados 4 classificadores sobre a mesma representação TF-IDF (unigramas+bigramas). "
          "Critério: **F1 macro em validação cruzada estratificada (5 folds)**, com conferência em "
          "hold-out de 20%. Em empate técnico (< 0,005) prevalece o modelo mais simples e interpretável.", ""]
    for tarefa, e in escolhidos.items():
        md += [f"## {tarefa.capitalize()}: {e['modelo']}", "",
               f"- F1 macro CV: **{e['f1_macro_cv']}** (±{e['desvio_cv']}) · hold-out F1: "
               f"**{e['f1_macro_holdout']}** · acurácia hold-out: **{e['acuracia_holdout']}**", "",
               _md_tabela(comp[comp.tarefa == tarefa].drop(columns="tarefa")), ""]
    md += ["## Por que esta abordagem", "",
           "- **Poucos dados e texto curto**: modelos lineares sobre TF-IDF costumam igualar ou superar "
           "modelos mais complexos, treinam em segundos e dispensam GPU.",
           "- **Interpretabilidade**: os coeficientes mostram quais termos puxam a decisão, "
           "o que ajuda a área comercial a confiar na recomendação.",
           "- **Probabilidades calibradas** alimentam diretamente o sistema de scoring (0-100).",
           "- **Cautela**: os dados atuais são sintéticos; os números absolutos são otimistas e devem "
           "ser reavaliados com as transcrições reais (ver `README.md`)."]
    (config.REPORTS_DIR / "modelo_selecionado.md").write_text("\n".join(md), encoding="utf-8")


def _md_tabela(df: pd.DataFrame) -> str:
    cab = "| " + " | ".join(df.columns) + " |"
    sep = "|" + "---|" * len(df.columns)
    corpo = ["| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join([cab, sep, *corpo])


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/scoring.py
"""Etapas de Indicadores e Recomendação.

Os scores (0-100) combinam a probabilidade do modelo (70%) com sinais léxicos
interpretáveis (30%). Os sinais léxicos tornam o score robusto a frases que o
modelo não viu no treino e explicam *por quê* o score subiu.
"""
import re

from .processing import limpar

PESO_MODELO, PESO_REGRA = 0.7, 0.3

# nome do sinal -> (regex sobre texto normalizado, peso). A soma dos pesos é limitada a 1.
SINAIS_CHURN = {
    "cancelar": (r"cancel\w*", 0.6), "encerrar": (r"encerr\w*", 0.6),
    "portabilidade": (r"portabilidade", 0.6), "quer sair": (r"(vou|quero|vamos) sair|sair se", 0.6),
    "migrar/mudar de empresa": (r"(migrar|migracao|mudar para outr\w*)", 0.5),
    "concorrência": (r"(concorrenci\w*|concorrente\w*)", 0.4),
    "cotou/foi para outra empresa": (r"(cotei|fui) (em|na|no|para) outr\w*|outr[ao] (banco|empresa|instituicao|operadora|seguradora|lugar)", 0.4),
    "não pretende continuar": (r"nao pretendo continuar", 0.6),
    "reclamação recorrente": (r"(terceira|quarta|varias) vez\w*|ja liguei", 0.2),
}
NEGACAO_CHURN = re.compile(r"nao (quero|vou|pretendo) (cancelar|sair|encerrar)|prefiro continuar")

SINAIS_OPORT = {
    "interesse declarado": (r"(tenho interesse|queria conhecer|gostaria de conhecer|adorei)", 0.4),
    "contratar": (r"contrat\w+", 0.4), "cotação/proposta": (r"(cotacao|proposta)", 0.5),
    "aumentar limite/cobertura": (r"(aumentar|ampliar) (o )?(limite|cobertura)|incluir", 0.6),
    "quer investir": (r"(quero|queria|gostaria de) investir|investir mais|melhor rentabilidade", 0.6),
    "mais benefícios": (r"(mais beneficios|cashback|programa de pontos|previdencia|diversificar)", 0.4),
}

# léxico simples de sentimento (complementa o modelo em vocabulário não visto no treino)
LEX_NEG = r"(chateado|absurdo|irritad\w*|decepcionad\w*|abusiv\w*|pessim\w*|revoltad\w*|insatisfeit\w*|travand\w*|reclam\w*|cobrad\w* (indevid\w*|duas vezes)|nao (abre|funciona|resolve\w*)|mais cara|dificil|descaso|ninguem (resolve|avisou|me avis\w*))"
LEX_POS = r"(agradecer|otimo|otima|excelente|adorei|adoro|parabens|satisfeit\w*|feliz|contente|recomendo|rapid\w*|perfeito|gostei|tudo otimo|super)"


RECOMENDACOES = {
    "retencao_urgente": "Acionar o time de retenção em até 24h: ligar com oferta personalizada "
                        "(isenção de tarifa, revisão de taxa ou bônus) e tratar a causa-raiz da reclamação.",
    "retencao_preventiva": "Cliente com sinais de insatisfação: fazer follow-up em 7 dias, confirmar que o problema "
                           "foi resolvido e oferecer um benefício de fidelização.",
    "oportunidade": "Encaminhar o lead ao time comercial com a oferta indicada abaixo "
                    "e agendar contato em até 48h.",
    "oportunidade_com_risco": "Cliente tem interesse comercial mas também risco de saída: resolver primeiro a "
                              "insatisfação e só depois apresentar a oferta.",
    "acompanhar": "Sem sinal relevante de risco ou venda: registrar o atendimento e manter o relacionamento regular.",
    "elogio": "Cliente satisfeito: pedir indicação/avaliação (NPS) e registrar como caso de sucesso.",
}

OFERTA_POR_PRODUTO = {
    "Conta Digital": "pacote de serviços / cartão vinculado à conta",
    "Cartão de Crédito": "aumento de limite ou cartão com mais benefícios",
    "Empréstimo": "refinanciamento ou novo crédito com taxa menor",
    "Seguro": "seguro complementar (residencial/vida) ou ampliação de cobertura",
    "Investimentos": "produto de maior rentabilidade ou previdência",
}


LIMIAR_CHURN, LIMIAR_OPORT = 35, 50  # limiares de alerta (ver README)


def sinais(txt: str, dicionario: dict) -> list[tuple[str, float]]:
    t = limpar(txt)
    return [(nome, w) for nome, (rx, w) in dicionario.items() if re.search(rf"\b(?:{rx})\b", t)]


def nivel(score: float) -> str:
    return "Alto" if score >= 65 else "Médio" if score >= 35 else "Baixo"


def _combinar(p_modelo: float, achados: list[tuple[str, float]]) -> float:
    regra = min(1.0, sum(w for _, w in achados))
    return round(100 * (PESO_MODELO * p_modelo + PESO_REGRA * regra), 1)


def score_churn(p_modelo: float, txt: str) -> tuple[float, list[str]]:
    achados = sinais(txt, SINAIS_CHURN)
    if NEGACAO_CHURN.search(limpar(txt)):  # "não quero cancelar" não é intenção de saída
        achados = [(n, w) for n, w in achados if n not in ("cancelar", "encerrar", "quer sair")]
        return min(30.0, _combinar(p_modelo, achados)), [n for n, _ in achados]
    return _combinar(p_modelo, achados), [n for n, _ in achados]


def score_oportunidade(p_modelo: float, txt: str) -> tuple[float, list[str]]:
    achados = sinais(txt, SINAIS_OPORT)
    return _combinar(p_modelo, achados), [n for n, _ in achados]


def probas_lexico(txt: str) -> dict[str, float]:
    t = limpar(txt)
    neg = len(re.findall(rf"\b(?:{LEX_NEG})\b", t))
    pos = len(re.findall(rf"\b(?:{LEX_POS})\b", t))
    raw = {"negativo": neg, "positivo": pos, "neutro": 0.7}
    tot = sum(raw.values())
    return {k: v / tot for k, v in raw.items()}


def mistura_sentimento(p_modelo: dict[str, float], txt: str, peso_lexico: float = 0.4) -> dict[str, float]:
    lex = probas_lexico(txt)
    return {k: (1 - peso_lexico) * p_modelo.get(k, 0) + peso_lexico * lex[k] for k in lex}


def score_sentimento(probas: dict[str, float]) -> float:
    """-100 (muito negativo) a +100 (muito positivo)."""
    return round(100 * (probas.get("positivo", 0) - probas.get("negativo", 0)), 1)


def recomendar(produto: str, sentimento: str, s_churn: float, s_oport: float) -> tuple[str, str, str]:
    """Retorna (prioridade, ação, oferta sugerida)."""
    oferta = OFERTA_POR_PRODUTO.get(produto, "")
    if s_churn >= 65:
        return "Crítica", RECOMENDACOES["retencao_urgente"], ""
    if s_oport >= 65 and s_churn >= 35:
        return "Alta", RECOMENDACOES["oportunidade_com_risco"], oferta
    if s_oport >= 65:
        return "Alta", RECOMENDACOES["oportunidade"], oferta
    if s_churn >= 35 or sentimento == "negativo":
        return "Média", RECOMENDACOES["retencao_preventiva"], ""
    if s_oport >= 35:
        return "Média", RECOMENDACOES["oportunidade"], oferta
    if sentimento == "positivo":
        return "Baixa", RECOMENDACOES["elogio"], ""
    return "Baixa", RECOMENDACOES["acompanhar"], ""

In [ ]:
%%writefile src/pipeline.py
"""Pipeline completo: Transcrição -> Processamento -> Modelo -> Indicadores -> Recomendação."""
import json
from functools import lru_cache

import joblib
import numpy as np

from . import config
from .processing import termos_frequentes, texto_processado
from .scoring import (mistura_sentimento, nivel, recomendar, score_churn,
                      score_oportunidade, score_sentimento)


@lru_cache(maxsize=1)
def _carregar():
    if not config.MODELO_PATH.exists():
        raise FileNotFoundError("Modelos não treinados. Rode: python -m src.train")
    return joblib.load(config.MODELO_PATH)["modelos"]


def _proba(modelo, texto: str) -> dict:
    p = modelo.predict_proba([texto])[0]
    return {str(c): float(v) for c, v in zip(modelo.classes_, p)}


def termos_chave(txt: str, modelo_prod, n: int = 8) -> list[str]:
    """Termos com maior peso TF-IDF no documento (mistura frequência e raridade no corpus)."""
    tfidf = modelo_prod.named_steps["tfidf"]
    vet = tfidf.transform([texto_processado(txt)])
    vocab = np.array(tfidf.get_feature_names_out())
    idx = vet.toarray()[0].argsort()[::-1][:n]
    return [str(vocab[i]) for i in idx if vet[0, i] > 0]


def analisar(transcricao: str) -> dict:
    modelos = _carregar()
    texto = texto_processado(transcricao)

    p_prod = _proba(modelos["produto"], texto)
    produto = max(p_prod, key=p_prod.get)
    p_sent = mistura_sentimento(_proba(modelos["sentimento"], texto), transcricao)
    sentimento = max(p_sent, key=p_sent.get)
    p_churn = _proba(modelos["churn"], texto).get("1", 0.0)
    p_op = _proba(modelos["oportunidade"], texto).get("1", 0.0)

    s_churn, sin_churn = score_churn(p_churn, transcricao)
    s_op, sin_op = score_oportunidade(p_op, transcricao)
    prioridade, acao, oferta = recomendar(produto, sentimento, s_churn, s_op)

    return {
        "produto": produto, "confianca_produto": round(p_prod[produto], 3),
        "sentimento": sentimento, "score_sentimento": score_sentimento(p_sent),
        "risco_churn": s_churn, "nivel_churn": nivel(s_churn), "sinais_churn": sin_churn,
        "oportunidade": s_op, "nivel_oportunidade": nivel(s_op), "sinais_oportunidade": sin_op,
        "principais_termos": termos_chave(transcricao, modelos["produto"]),
        "termos_frequentes": [t for t, _ in termos_frequentes(transcricao, 5)],
        "prioridade": prioridade, "recomendacao": acao, "oferta_sugerida": oferta,
    }


def formatar(res: dict) -> str:
    return "\n".join([
        f"Produto identificado : {res['produto']} (confiança {res['confianca_produto']:.0%})",
        f"Sentimento           : {res['sentimento']} (score {res['score_sentimento']:+.0f})",
        f"Risco de churn       : {res['risco_churn']:.0f}/100 [{res['nivel_churn']}]"
        + (f"  sinais: {', '.join(res['sinais_churn'])}" if res["sinais_churn"] else ""),
        f"Oportunidade comercial: {res['oportunidade']:.0f}/100 [{res['nivel_oportunidade']}]"
        + (f"  sinais: {', '.join(res['sinais_oportunidade'])}" if res["sinais_oportunidade"] else ""),
        f"Principais termos    : {', '.join(res['principais_termos'])}",
        f"Prioridade           : {res['prioridade']}",
        f"Recomendação         : {res['recomendacao']}"
        + (f"\nOferta sugerida      : {res['oferta_sugerida']}" if res["oferta_sugerida"] else ""),
    ])


if __name__ == "__main__":
    import sys
    txt = open(sys.argv[1], encoding="utf-8").read() if len(sys.argv) > 1 else sys.stdin.read()
    print(formatar(analisar(txt)))

In [ ]:
%%writefile src/validate.py
"""Validação do pipeline completo em transcrições novas (nunca vistas no treino).

Uso: python -m src.validate
Saídas: reports/validacao.md, reports/validacao_detalhe.csv
"""
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

from . import config
from .pipeline import analisar
from .scoring import LIMIAR_CHURN, LIMIAR_OPORT

# como converter a saída do pipeline em classe prevista
PRED = {
    "produto": lambda r: r["produto"],
    "sentimento": lambda r: r["sentimento"],
    "churn": lambda r: int(r["risco_churn"] >= LIMIAR_CHURN),
    "oportunidade": lambda r: int(r["oportunidade"] >= LIMIAR_OPORT),
}


def avaliar(df: pd.DataFrame, nome: str):
    res = [analisar(t) for t in df["transcricao"]]
    linhas = {}
    for tarefa, col in config.TAREFAS.items():
        pred = [PRED[tarefa](r) for r in res]
        linhas[tarefa] = {
            "acuracia": round(accuracy_score(df[col], pred), 3),
            "f1_macro": round(f1_score(df[col], pred, average="macro", zero_division=0), 3)}
    detalhe = df.copy()
    for tarefa in config.TAREFAS:
        detalhe[f"pred_{tarefa}"] = [PRED[tarefa](r) for r in res]
    detalhe["risco_churn"] = [r["risco_churn"] for r in res]
    detalhe["score_oportunidade"] = [r["oportunidade"] for r in res]
    detalhe["prioridade"] = [r["prioridade"] for r in res]
    return linhas, detalhe


def main():
    partes = ["# Validação com novas transcrições", ""]
    for nome, path, aviso in [
        ("Manuais (escritas à mão)", config.NOVAS_MANUAIS_CSV,
         "As regras do scoring foram calibradas olhando estes casos: resultado NÃO é independente."),
        ("Cegas (escritas à mão, nunca usadas para ajuste)", config.NOVAS_CEGAS_CSV,
         "Melhor estimativa de generalização disponível; amostra pequena, então a variância é alta."),
        ("Sintéticas (seed diferente do treino)", config.NOVAS_SINTETICAS_CSV,
         "Mesma distribuição do treino, portanto resultado otimista; serve como teste de regressão."),
    ]:
        if not path.exists():
            continue
        df = pd.read_csv(path)
        linhas, detalhe = avaliar(df, nome)
        slug = path.stem.replace("novas_", "")
        detalhe.to_csv(config.REPORTS_DIR / f"validacao_detalhe_{slug}.csv", index=False)
        partes += [f"## {nome}", "", f"_{aviso}_  (n = {len(df)})", "",
                   "| Tarefa | Acurácia | F1 macro |", "|---|---|---|"]
        partes += [f"| {t} | {m['acuracia']} | {m['f1_macro']} |" for t, m in linhas.items()]
        erros = detalhe[(detalhe.produto != detalhe.pred_produto) | (detalhe.sentimento != detalhe.pred_sentimento)
                        | (detalhe.churn != detalhe.pred_churn) | (detalhe.oportunidade != detalhe.pred_oportunidade)]
        partes += ["", f"Transcrições com ao menos um erro: **{len(erros)} de {len(df)}**.", ""]
        print(f"\n== {nome} ==")
        for t, m in linhas.items():
            print(f"  {t:13s} acc={m['acuracia']}  f1={m['f1_macro']}")
    config.REPORTS_DIR.mkdir(exist_ok=True)
    (config.REPORTS_DIR / "validacao.md").write_text("\n".join(partes), encoding="utf-8")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/totvs_catalog.py
# Copiado de Guiroselli/TOTUVsVers-oFinal (backend/totvs_catalog.py, v2.1.0). Só o import de normalization foi tornado local.
"""
Catálogo detalhado de produtos TOTVS e Motor Híbrido de Recomendação Explicável.
Combina regras determinísticas, cobertura de dores, tarefas, urgência, tabela explícita de segmentos e tratamento de negações.
"""

import re
from typing import List, Dict, Any, Optional, Tuple

CATALOG_VERSION = "2.1.0"

# Tabela explícita de segmentos corporativos e capacidades
SEGMENT_MAP: Dict[str, List[str]] = {
    "MANUFATURA": ["protheus", "supply", "analytics"],
    "SERVICOS": ["fluig", "crm", "rh"],
    "SAUDE": ["rh", "protheus", "fluig"],
    "LOGISTICA": ["supply", "protheus", "analytics"],
    "VAREJO": ["crm", "protheus", "analytics"],
    "FINANCEIRO": ["analytics", "protheus", "fluig"],
    "AGRO": ["protheus", "supply", "analytics"],
    "EDUCACIONAL": ["fluig", "rh", "analytics"],
    "CONSTRUCAO": ["protheus", "supply", "fluig"],
    "DISTRIBUICAO": ["supply", "protheus", "crm"],
}

# Palavras genéricas que NÃO geram recomendação sozinhas
PALAVRAS_GENERICAS = {
    "sistema", "cliente", "reunião", "reuniao", "processo", "problema", 
    "coisa", "assunto", "pauta", "projeto", "demanda", "ajuste", "verificar",
    "falar", "conversar", "alinhar", "revisar", "acompanhar"
}

# Padrões de negação que anulam gatilhos
NEGATION_PATTERNS = [
    r"n[aã]o\s+(?:temos|h[aá]|tivemos|estamos\s+com)\s+(?:nenhum\s+)?problema\s+(?:de|com|em)\s+([a-zá-ú\s]+)",
    r"sem\s+(?:nenhum\s+)?problema\s+(?:de|com|em)\s+([a-zá-ú\s]+)",
    r"j[aá]\s+(?:est[aá]\s+)?resolvido\s+(?:o|a|os|as)?\s+([a-zá-ú\s]+)",
    r"n[aã]o\s+h[aá]\s+(?:gargalo|dificuldade|atraso)\s+(?:em|no|na)\s+([a-zá-ú\s]+)",
    r"n[aã]o\s+(?:precisa|precisamos|h[aá]\s+necessidade)\s+de\s+([a-zá-ú\s]+)",
    r"n[aã]o\s+temos\s+interesse\s+em\s+([a-zá-ú\s]+)",
    r"est[aá]\s+(?:100%|totalmente)\s+(?:garantido|resolvido|estruturado)\s+([a-zá-ú\s]+)",
]

CATALOGO_TOTVS: Dict[str, Dict[str, Any]] = {
    "fluig": {
        "product_key": "fluig",
        "product_name": "TOTVS Fluig",
        "product_area": "Workflow, BPM, ECM e Portais Corporativos",
        "capacidades": [
            "Fluxos formais de aprovação multi-alçada com SLA",
            "Gestão e versionamento eletrônico de documentos (ECM)",
            "Rastreabilidade ponta a ponta de processos e etapas",
            "Notificações automáticas e delegação de responsáveis por tarefa"
        ],
        "dores_atendidas": ["aprovacao_pendente", "documento_faltando"],
        "palavras_fortes": [
            "aprovação", "aprovacao", "alçada", "alcada", "workflow", "fluxo de aprovação", 
            "ecm", "bpm", "assinatura eletrônica", "assinatura eletronica", "documento pendente", 
            "protocolo formal", "parecer formal", "validação de contrato"
        ],
        "palavras_genericas": ["contrato", "documento", "validar", "assinar", "fluxo", "revisão", "revisao"],
        "beneficios_esperados": [
            "Pode reduzir aprovações informais e descentralizadas por e-mail",
            "Dá visibilidade total do responsável e prazos de cada etapa em tempo real",
            "Contribui para eliminar extravios de documentos e perda de prazos regulatórios"
        ],
        "pre_requisitos": [
            "Mapeamento do processo atual e regras de alçada",
            "Definição da matriz de aprovadores por departamento",
            "Validação dos formulários e dados necessários em cada etapa"
        ],
        "dados_necessarios": ["tarefas_confirmadas", "responsavel", "prazo", "codigo_cliente"],
        "proximo_passo_implantacao": "Mapear processo, aprovadores e criar fluxo piloto no TOTVS Fluig.",
        "referencia_oficial": "https://www.totvs.com/fluig/"
    },
    "crm": {
        "product_key": "crm",
        "product_name": "TOTVS CRM Gestão de Clientes",
        "product_area": "Gestão Comercial, Oportunidades e Retenção",
        "capacidades": [
            "Visão 360° do cliente e registro centralizado de interações",
            "Gestão do pipeline de vendas e oportunidades",
            "Acompanhamento sistemático de follow-ups comerciais e pós-venda",
            "Alertas preventivos de insatisfação e risco de cancelamento (churn)"
        ],
        "dores_atendidas": ["insatisfacao_cliente", "atraso_prazo"],
        "palavras_fortes": [
            "pipeline", "proposta comercial", "follow-up", "followup", "churn", "retenção", "retencao",
            "carteira de clientes", "visita comercial", "insatisfação do cliente", "insatisfacao do cliente", 
            "pós-venda", "pos-venda", "funil de vendas", "perda de cliente"
        ],
        "palavras_genericas": ["cliente", "contato", "reunião", "reuniao", "retorno", "visita", "comercial", "satisfação"],
        "beneficios_esperados": [
            "Pode apoiar na retenção de clientes insatisfeitos com atendimento estruturado",
            "Assegura que nenhum compromisso ou follow-up comercial seja esquecido",
            "Centraliza o histórico de relacionamento acessível a toda a equipe"
        ],
        "pre_requisitos": [
            "Cadastro e segmentação atualizada da base de clientes",
            "Estruturação das etapas do funil de relacionamento",
            "Definição de responsáveis por conta/carteira"
        ],
        "dados_necessarios": ["codigo_cliente", "tarefas_confirmadas", "historico_contato"],
        "proximo_passo_implantacao": "Vincular tarefa ao cliente no CRM e criar atividade comercial de follow-up.",
        "referencia_oficial": "https://www.totvs.com/crm/"
    },
    "protheus": {
        "product_key": "protheus",
        "product_name": "TOTVS Protheus (ERP)",
        "product_area": "ERP Backoffice, Gestão Financeira e Operações",
        "capacidades": [
            "Gestão financeira integrada (Contas a Pagar/Receber, Tesouraria e Fluxo de Caixa)",
            "Faturamento, emissão de notas fiscais e conformidade tributária",
            "Planejamento e controle de compras e suprimentos (SIGACOM)",
            "Controle operacional, produção (PCP) e custos integrados"
        ],
        "dores_atendidas": ["gargalo_operacional", "atraso_prazo", "gestao_estoque"],
        "palavras_fortes": [
            "faturamento", "emissão de nf", "emissao de nf", "nota fiscal", "contas a pagar", "contas a receber",
            "conciliação bancária", "conciliacao bancaria", "pcp", "sigacom", "sigafin", "sigafat",
            "fechamento contábil", "fechamento contabil", "tributário", "tributario", "protheus"
        ],
        "palavras_genericas": ["sistema", "pedido", "pagamento", "financeiro", "nf", "erp", "operação", "operacao", "custo", "compra"],
        "beneficios_esperados": [
            "É compatível com a automação de rotinas fiscais, contábeis e financeiras",
            "Ajuda a mitigar gargalos operacionais e retrabalho manual",
            "Proporciona rastreabilidade e integridade aos dados de faturamento e compras"
        ],
        "pre_requisitos": [
            "Identificação do módulo ERP afetado (ex: SIGAFIN, SIGACOM, SIGAEST, SIGAFAT)",
            "Validação dos endpoints da API REST do Protheus",
            "Alinhamento dos parâmetros e dicionário de dados"
        ],
        "dados_necessarios": ["modulo_protheus", "tarefas_confirmadas", "codigo_cliente"],
        "proximo_passo_implantacao": "Identificar módulo, processo impactado e validar endpoint/API no Protheus.",
        "referencia_oficial": "https://www.totvs.com/protheus/"
    },
    "analytics": {
        "product_key": "analytics",
        "product_name": "TOTVS Analytics & Fast Analytics",
        "product_area": "Business Intelligence, Dashboards e Gestão à Vista",
        "capacidades": [
            "Dashboards executivos consolidados com atualização dinâmica",
            "Indicadores de desempenho (KPIs), metas e alertas de desvio",
            "Análise de evolução temporal, tendências e séries históricas",
            "Visão analítica por cliente, segmento, unidade e produto"
        ],
        "dores_atendidas": ["falta_metricas", "falta_visibilidade"],
        "palavras_fortes": [
            "dashboard", "kpi", "indicador de desempenho", "painel analítico", "painel analitico",
            "fast analytics", "business intelligence", "métrica gerencial", "metrica gerencial",
            "visão analítica", "visao analitica", "relatório de indicadores", "relatorio de indicadores", "bi"
        ],
        "palavras_genericas": ["relatório", "relatorio", "métrica", "metrica", "indicador", "medir", "dados", "visão", "visao", "painel", "gráfico", "grafico"],
        "beneficios_esperados": [
            "Pode apoiar na rápida visualização de gargalos e metas corporativas",
            "Substitui relatórios manuais em planilhas dispersas por painéis automatizados",
            "Proporciona embasamento orientado a dados para tomada de decisões estratégicas"
        ],
        "pre_requisitos": [
            "Definição dos KPIs e métricas prioritárias para o negócio",
            "Mapeamento das fontes de dados (ERP, CRM, planilhas)",
            "Definição da periodicidade e perfis de acesso"
        ],
        "dados_necessarios": ["metricas_alvo", "fontes_dados", "periodicidade"],
        "proximo_passo_implantacao": "Definir indicadores prioritários, fontes de dados e periodicidade no Analytics.",
        "referencia_oficial": "https://www.totvs.com/analytics/"
    },
    "rh": {
        "product_key": "rh",
        "product_name": "TOTVS RH (Linha RM / Protheus RH / Meu RH)",
        "product_area": "Gestão de Pessoas, Folha, Ponto e Recrutamento",
        "capacidades": [
            "Folha de pagamento automatizada e atendimento ao eSocial",
            "Gestão de ponto eletrônico, escalas e controle de jornada",
            "Recrutamento, seleção digital e onboarding de colaboradores",
            "Avaliação de desempenho, gestão de clima e capacitação"
        ],
        "dores_atendidas": ["problemas_equipe"],
        "palavras_fortes": [
            "folha de pagamento", "ponto eletrônico", "ponto eletronico", "escala de trabalho", 
            "recrutamento", "onboarding", "esocial", "meu rh", "avaliação de desempenho", 
            "avaliacao de desempenho", "admissão", "admissao", "afastamento", "atestado médico", 
            "atestado medico", "homologação de atestado", "homologacao de atestado", "sobreaviso"
        ],
        "palavras_genericas": ["contratar", "contratação", "contratacao", "equipe", "treinamento", "folha", "pessoal", "escala", "vaga", "rh", "hora extra", "férias", "ferias", "sobrecarga"],
        "beneficios_esperados": [
            "Pode auxiliar no dimensionamento adequado da carga de trabalho da equipe",
            "Agiliza processos seletivos para reposição de quadros e redução de sobrecarga",
            "Garante conformidade trabalhista com gestão digital de jornada e ponto"
        ],
        "pre_requisitos": [
            "Mapeamento da necessidade de headcount ou treinamento",
            "Classificação da demanda (recrutamento, folha, capacitação, escalas)",
            "Aprovação de alçadas de contratação"
        ],
        "dados_necessarios": ["tipo_demanda_rh", "departamento", "tarefas_confirmadas"],
        "proximo_passo_implantacao": "Classificar a demanda e encaminhar ao processo de RH adequado (recrutamento/gestão).",
        "referencia_oficial": "https://www.totvs.com/rh/"
    },
    "supply": {
        "product_key": "supply",
        "product_name": "TOTVS Supply Chain & WMS",
        "product_area": "Gestão de Armazenagem, WMS e Logística",
        "capacidades": [
            "Endereçamento inteligente e gestão operacional de armazéns (WMS)",
            "Controle de inventário físico vs sistêmico em tempo real",
            "Rastreamento de pedidos, expedição e agendamento de entregas",
            "Gestão de fretes e integração com transportadoras"
        ],
        "dores_atendidas": ["gestao_estoque", "atraso_prazo"],
        "palavras_fortes": [
            "wms", "endereçamento inteligente", "enderecamento inteligente", "inventário físico", 
            "inventario fisico", "ruptura de estoque", "gestão de armazém", "gestao de armazem", 
            "expedição e entrega", "expedicao e entrega", "conferência de carga", "conferencia de carga", 
            "tabela de frete", "romaneio", "balança integrada", "balanca integrada"
        ],
        "palavras_genericas": ["estoque", "armazém", "armazem", "entrega", "logística", "logistica", "transporte", "expedição", "expedicao", "material", "fornecedor", "ruptura", "inventário", "inventario"],
        "beneficios_esperados": [
            "Pode prevenir rupturas de estoque e desvios de inventário",
            "Otimiza o fluxo de separação, conferência e expedição de mercadorias",
            "Melhora a pontualidade das entregas e o nível de serviço ao cliente"
        ],
        "pre_requisitos": [
            "Mapeamento do layout de armazenagem e endereçamento",
            "Definição de regras de estocagem (FIFO, FEFO, lote)",
            "Identificação das integrações com transportadoras e ERP"
        ],
        "dados_necessarios": ["tipo_operacao_logistica", "unidade_armazenagem", "tarefas_confirmadas"],
        "proximo_passo_implantacao": "Mapear o processo logístico e identificar módulo de WMS ou integração de fretes.",
        "referencia_oficial": "https://www.totvs.com/supply-chain/"
    }
}


def check_negation(text: str, target_term: str) -> bool:
    """Verifica se o termo alvo está sob efeito de uma negação na frase."""
    if not text:
        return False
    text_lower = text.lower()
    for pat in NEGATION_PATTERNS:
        match = re.search(pat, text_lower)
        if match:
            negated_clause = match.group(0)
            terms = re.split(r"[\s_]+", target_term.lower())
            if target_term.lower() in negated_clause or any(len(w) >= 3 and w in negated_clause for w in terms):
                return True
    return False


def compute_recommendations(
    dores: List[Any],
    tarefas: List[Any],
    tema: str = "",
    urgencia: str = "Média",
    client_code: str = "",
    segmento: str = "",
    integracoes_config: Optional[Dict[str, Any]] = None,
    persisted_recommendations: Optional[Dict[str, Any]] = None,
    full_transcript: str = ""
) -> List[Dict[str, Any]]:
    """
    Motor híbrido explicável de recomendação de produtos TOTVS calibrado.
    Calcula fit_score combinando:
    - 35% Cobertura de Dores (pain_coverage)
    - 25% Compatibilidade de Tarefas (task_fit)
    - 15% Impacto da Urgência (urgency_weight)
    - 15% Adequação ao Segmento/Tema (segment_fit)
    - 10% Disponibilidade da Integração (integration_availability)
    """
    if integracoes_config is None:
        integracoes_config = {}
    if persisted_recommendations is None:
        persisted_recommendations = {}

    # 1. Normaliza dores e evidências
    categorias_dores = []
    dores_com_evidencia = []
    for d in dores:
        if isinstance(d, dict):
            cat = str(d.get("categoria", "outro")).lower()
            categorias_dores.append(cat)
            trecho = d.get("trecho") or d.get("descricao") or ""
            dores_com_evidencia.append({
                "categoria": cat,
                "label": d.get("label") or cat.replace("_", " ").title(),
                "evidence": str(trecho)[:250]
            })
        elif isinstance(d, str):
            normalize_pain_category = lambda s: re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")
            cat = normalize_pain_category(d)
            categorias_dores.append(cat)
            dores_com_evidencia.append({
                "categoria": cat,
                "label": cat.replace("_", " ").title(),
                "evidence": d[:250]
            })

    total_dores = len(categorias_dores) or 1

    # 2. Normaliza tarefas
    tarefas_lista = []
    for t in tarefas:
        if isinstance(t, dict):
            texto = str(t.get("tarefa", "")).lower()
            tarefas_lista.append({
                "task": t.get("tarefa", ""),
                "responsible": t.get("responsavel", "Não identificado"),
                "due_date": t.get("prazo", "Não mencionado"),
                "texto_lower": texto
            })

    total_tarefas = len(tarefas_lista) or 1

    # 3. Determina segmento mapeado (sem correspondência por letras soltas)
    segmento_detectado = (segmento or "").upper().strip()
    if not segmento_detectado and client_code:
        # Se client_code for um nome de segmento (ex: 'MANUFATURA', 'SAUDE')
        code_upper = client_code.upper().strip()
        if code_upper in SEGMENT_MAP:
            segmento_detectado = code_upper

    produtos_do_segmento = SEGMENT_MAP.get(segmento_detectado, [])

    recommendations = []

    for key, prod in CATALOGO_TOTVS.items():
        contributing_signals = []
        negated_signals = []

        # 1. Cobertura de dores (35%)
        dores_match = []
        for d in dores_com_evidencia:
            if d["categoria"] in prod["dores_atendidas"]:
                # Verifica negação na evidência
                if d["evidence"] and check_negation(d["evidence"], d["categoria"]):
                    negated_signals.append(f"Dor '{d['label']}' ignorada por negação no texto.")
                else:
                    dores_match.append(d)
                    contributing_signals.append(f"Dor mapeada: {d['label']}")

        pain_coverage = len(dores_match) / total_dores if dores_com_evidencia else 0.0

        # 2. Compatibilidade de tarefas (25%) com separação de termos fortes vs genéricos
        tarefas_match = []
        for t in tarefas_lista:
            # Termos fortes têm prioridade e peso integral
            forte_encontrado = any(kw in t["texto_lower"] for kw in prod["palavras_fortes"])
            generico_encontrado = any(kw in t["texto_lower"] for kw in prod["palavras_genericas"])

            if forte_encontrado or (generico_encontrado and dores_match):
                # Verifica se a tarefa contém negação
                if check_negation(t["texto_lower"], key):
                    negated_signals.append(f"Tarefa '{t['task']}' ignorada por negação contextual.")
                else:
                    tarefas_match.append({
                        "task": t["task"],
                        "responsible": t["responsible"],
                        "due_date": t["due_date"],
                        "strong_match": forte_encontrado
                    })
                    contributing_signals.append(f"Tarefa compatível: {t['task']}")

        # Penaliza se foram apenas palavras genéricas sem correspondência forte
        task_fit_raw = len(tarefas_match) / total_tarefas if tarefas_lista else 0.0
        has_strong_task = any(t.get("strong_match") for t in tarefas_match)
        task_fit = task_fit_raw if (has_strong_task or dores_match) else (task_fit_raw * 0.4)

        # 3. Impacto da urgência (15%)
        urgencia_weight = 0.0
        if urgencia in ["Crítica", "Critica", "Alta"]:
            if dores_match or has_strong_task:
                urgencia_weight = 1.0 if urgencia.startswith("Cr") else 0.8
                contributing_signals.append(f"Urgência {urgencia} elevou a prioridade do módulo")
        elif urgencia == "Média" and (dores_match or has_strong_task):
            urgencia_weight = 0.5
        elif dores_match or has_strong_task:
            urgencia_weight = 0.2

        # 4. Adequação ao Segmento e Tema (15%)
        segment_fit = 0.0
        tema_lower = (tema or "").lower()
        tema_forte = any(kw in tema_lower for kw in prod["palavras_fortes"])
        
        if key in produtos_do_segmento:
            segment_fit += 0.6
            contributing_signals.append(f"Módulo prioritário para o segmento {segmento_detectado}")
        elif not segmento_detectado:
            # Segmento não informado: adequação zero para evitar falsos positivos
            contributing_signals.append("Segmento do cliente não informado / não mapeado")
        
        if tema_forte:
            segment_fit += 0.4
            contributing_signals.append(f"Tema da reunião alinhado diretamente ao módulo: {tema}")

        segment_fit = min(1.0, segment_fit)

        # 5. Disponibilidade da Integração (10%)
        is_configurado = bool(integracoes_config.get(key, {}).get("webhook_url"))
        integration_avail = 1.0 if is_configurado else 0.5
        if is_configurado:
            contributing_signals.append("Webhook oficial configurado no ecossistema")

        # 6. Cálculo do Fit Score Ponderado (5 fatores estritos)
        fit_score = (
            (0.35 * pain_coverage) +
            (0.25 * task_fit) +
            (0.15 * urgencia_weight) +
            (0.15 * segment_fit) +
            (0.10 * integration_avail)
        )

        # Se não houver dores nem tarefas nem tema forte, penaliza expressivamente
        if not dores_match and not has_strong_task and not tema_forte:
            fit_score = round(fit_score * 0.15, 2)
        else:
            fit_score = round(min(1.0, max(0.0, fit_score)), 2)

        # 7. Calibração da Confiança (independente de palavra isolada)
        signals_count = len(contributing_signals)
        seen_ev = set()
        evidences_list = []
        for d in dores_match:
            ev = (d.get("evidence") or d.get("trecho") or "").strip()
            if ev and ev.lower() not in seen_ev:
                seen_ev.add(ev.lower())
                evidences_list.append(ev)

        # Agrupa e deduplica dores atendidas para a justificativa e trigger_pains
        grouped_match = {}
        for d in dores_match:
            cat = d["categoria"]
            if cat not in grouped_match:
                grouped_match[cat] = {
                    "categoria": cat,
                    "label": d.get("label") or cat.replace("_", " ").title(),
                    "evidencias": []
                }
            ev = (d.get("evidence") or d.get("trecho") or "").strip()
            if ev and ev not in grouped_match[cat]["evidencias"]:
                grouped_match[cat]["evidencias"].append(ev)
                
        dedup_trigger_pains = []
        for cat, gm in grouped_match.items():
            cnt = len(gm["evidencias"]) or 1
            dedup_trigger_pains.append({
                "categoria": cat,
                "label": f"{gm['label']} — {cnt} ocorrências" if cnt > 1 else gm["label"],
                "ocorrencias": cnt,
                "evidence": " | ".join(gm["evidencias"][:2]) if gm["evidencias"] else "Mapeada em conformidade com o módulo."
            })
        
        # Base de confiança: qualidade de evidências e múltiplos sinais
        confidence = 0.20
        if evidences_list:
            avg_ev_len = sum(len(e) for e in evidences_list) / len(evidences_list)
            if avg_ev_len > 30:
                confidence += 0.30
            else:
                confidence += 0.15
        if signals_count >= 3:
            confidence += 0.30
        elif signals_count >= 2:
            confidence += 0.15
        if has_strong_task:
            confidence += 0.15
        if dores_match and has_strong_task:
            confidence += 0.10
        if not segmento_detectado:
            confidence -= 0.05
        if negated_signals:
            confidence -= 0.20

        confidence = round(min(0.98, max(0.10, confidence)), 2)

        is_strong = (confidence >= 0.50 and fit_score >= 0.40)
        confidence_label = "Alta" if confidence >= 0.75 else "Média" if confidence >= 0.50 else "Evidência insuficiente para recomendar com segurança"

        # 8. Justificativa Explicável com Deduplicação e Contexto do Módulo
        motivos = []
        if dedup_trigger_pains:
            pain_phrases = [p["label"] for p in dedup_trigger_pains[:2]]
            motivos.append(f"A reunião registrou demandas relacionadas a {', '.join(pain_phrases)}.")

        if tarefas_match:
            tarefas_txt = "1 tarefa operacionalmente compatível" if len(tarefas_match) == 1 else f"{len(tarefas_match)} tarefas operacionalmente compatíveis"
            motivos.append(f"Identificada {tarefas_txt} com as capacidades do módulo." if len(tarefas_match) == 1 else f"Identificadas {tarefas_txt} com as capacidades do módulo.")
        if urgencia in ["Crítica", "Alta"] and (dores_match or tarefas_match):
            motivos.append(f"O nível de urgência ({urgencia}) reforça a prioridade na estruturação deste processo.")
        if segmento_detectado and key in produtos_do_segmento:
            motivos.append(f"Solução homologada para o segmento {segmento_detectado}.")

        if not motivos:
            why = f"O produto {prod['product_name']} pode apoiar a operação como opção complementar (requer validação de escopo e da API)."
        else:
            why = " ".join(motivos) + f" O {prod['product_name']} pode apoiar a operação, sendo compatível com as rotinas identificadas (requer validação do módulo e da API)."

        persisted = persisted_recommendations.get(key, {})
        review_status = persisted.get("review_status", "pending")

        rec = {
            "product_key": key,
            "product_name": prod["product_name"],
            "product_area": prod["product_area"],
            "fit_score": fit_score,
            "confidence": confidence,
            "confidence_label": confidence_label,
            "is_strong_recommendation": is_strong,
            "why_recommended": why,
            "trigger_pains": dedup_trigger_pains,
            "trigger_tasks": tarefas_match,
            "all_evidences": evidences_list,
            "recommended_capabilities": prod["capacidades"],
            "expected_benefits": prod["beneficios_esperados"],
            "implementation_next_step": prod["proximo_passo_implantacao"],
            "required_inputs": prod["dados_necessarios"],
            "missing_data": [d for d in prod["dados_necessarios"] if d not in ["tarefas_confirmadas", "codigo_cliente"]] if not is_configurado else [],
            "integration_status": "real" if is_configurado else "simulated",
            "contributing_signals": contributing_signals,
            "negated_signals": negated_signals,
            "source": "rule_plus_catalog",
            "review_status": review_status,
            "catalog_version": CATALOG_VERSION,
            "score_breakdown": {
                "pain_coverage": round(pain_coverage, 2),
                "task_fit": round(task_fit, 2),
                "urgency_weight": round(urgencia_weight, 2),
                "segment_fit": round(segment_fit, 2),
                "integration_availability": round(integration_avail, 2)
            }
        }
        recommendations.append(rec)

    recommendations.sort(key=lambda x: x["fit_score"], reverse=True)
    filtradas = [r for r in recommendations if r["fit_score"] >= 0.20 and (r["trigger_pains"] or r["trigger_tasks"])]
    return filtradas[:3]

In [ ]:
%%writefile src/importar_totvs.py
"""Importa as reuniões reais do projeto Proton Flow (TOTUVsVers-oFinal).

Uso: python -m src.importar_totvs caminho/para/backend/dataset_limpo.json
Gera data/real/reunioes_totvs.json com as transcrições anonimizadas + a análise antiga (só para comparação).
"""
import hashlib
import json
import sys

from . import config

DESTINO = config.DATA_DIR / "real" / "reunioes_totvs.json"
MIN_CHARS = 300  # abaixo disso a reunião não tem transcrição útil


def converter(e: dict) -> dict:
    rec = (e.get("recomendacoes_totvs") or [])
    ia = e.get("RESUMO_IA") or {}
    return {
        "id": str(e["ID_MEETING"]),
        "data": str(e.get("DT_MEETING", ""))[:10],
        "segmento": e.get("NOME_SEGMENTO"),
        "duracao": e.get("DURACAO_MEETING"),
        "transcricao": e["ANON_TRANSCRICAO"],
        "hash": hashlib.md5(e["ANON_TRANSCRICAO"].encode("utf-8")).hexdigest()[:10],
        # análise antiga (LLM local + regras): usada apenas para comparação, NÃO é verdade absoluta
        "legado": {
            "tema": ia.get("tema"),
            "urgencia": e.get("NIVEL_URGENCIA"),
            "produto_top": rec[0]["product_key"] if rec else None,
            "dores": sorted({d.get("categoria") for d in ia.get("dores", []) if isinstance(d, dict)}),
        },
    }


def main():
    if len(sys.argv) < 2:
        raise SystemExit(__doc__)
    bruto = json.load(open(sys.argv[1], encoding="utf-8"))
    reunioes = [converter(e) for e in bruto if len(e.get("ANON_TRANSCRICAO") or "") >= MIN_CHARS]
    DESTINO.parent.mkdir(parents=True, exist_ok=True)
    json.dump(reunioes, open(DESTINO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    unicas = len({r["hash"] for r in reunioes})
    print(f"{len(reunioes)} reuniões com transcrição ({unicas} textos únicos) -> {DESTINO}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/totvs_pipeline.py
"""Pipeline para as reuniões reais do Proton Flow (transcrições longas, anonimizadas).

Transcrição -> Processamento -> Modelo (catálogo TOTVS + léxicos) -> Indicadores -> Recomendação

Diferenças para o pipeline sintético (src/pipeline.py):
- as reuniões têm de 7 mil a 120 mil caracteres e não há rótulos humanos; por isso o "modelo" de produto
  é o catálogo TOTVS (palavras fortes/genéricas por produto) e os indicadores usam sinais léxicos explicáveis;
- os termos principais vêm de TF-IDF ajustado no corpus real.
"""
import json
import math
import re
from functools import lru_cache

from sklearn.feature_extraction.text import TfidfVectorizer

from . import config
from .processing import STOPWORDS, limpar, remover_acentos
from .scoring import RECOMENDACOES, nivel
from .totvs_catalog import CATALOGO_TOTVS, compute_recommendations

REAL_JSON = config.DATA_DIR / "real" / "reunioes_totvs.json"

# ruído de transcrição oral e marcadores de anonimização
STOP_ORAL = set("""
locutor pessoa empresa local ta ne ai pra gente cara tipo assim entao ficar fica aqui ali la isso entendi beleza
sim nao ok certo ne so vou vai vamos ver vamos acho acha pode podemos tem tinha temos ter seria sendo todo toda
tudo nada alguma algum voces voce nos eles agora ainda depois antes ja entao tambem muito pouco mais menos
coisa coisas bom dia tarde noite obrigado obrigada ola
""".split())

# vocabulário de domínio que complementa as palavras do catálogo (acentos removidos)
EXTRAS_PRODUTO = {
    "fluig": ["fluig", "workflow", "aprovacao", "assinatura", "fluxo de aprovacao", "ecm"],
    "crm": ["crm", "oportunidade", "proposta", "renovacao", "follow up", "pos venda", "pipeline"],
    "protheus": ["protheus", "fiscal", "tributacao", "tributacao", "nota fiscal", "contabil", "faturamento",
                 "financeiro", "contas a pagar", "contas a receber", "fechamento", "nfe", "cte", "erp"],
    "analytics": ["dashboard", "kpi", "indicadores", "bi", "analytics", "relatorio gerencial"],
    "rh": ["folha", "ponto", "ferias", "atestado", "esocial", "admissao", "meu rh", "rm", "beneficios", "escala"],
    "supply": ["estoque", "armazem", "wms", "logistica", "expedicao", "frete", "inventario"],
}

REGRAS_DORES = {  # categoria -> (rótulo, regex sobre texto sem acentos)
    "aprovacao_pendente": ("Aprovação pendente", r"aguardando aprovacao|aprovacao pendente|falta aprovacao|depende (de|da) aprovacao|alcada"),
    "atraso_prazo": ("Atraso / prazo em risco", r"atras(o|ad[oa]s?)|fora do prazo|prazo (estourou|vencid\w*|curto)|nao (entregou|entregaram)|postergar"),
    "gargalo_operacional": ("Gargalo operacional", r"gargalo|retrabalho|planilha|travad[oa]|muito lent[oa]|nao consegu\w+|manualmente|erro de"),
    "insatisfacao_cliente": ("Insatisfação do cliente", r"insatisf\w+|reclam\w+|frustrad\w+|decepcion\w+|pessim[oa]|nao (estou|estamos) satisfeit\w+"),
    "documento_faltando": ("Documento não localizado", r"documento (faltando|pendente|nao localizado)|falta (o )?documento|extravi\w+"),
    "falta_metricas": ("Falta de métricas/visão", r"sem (indicador\w*|visibilidade|relatorio\w*)|nao temos (indicador\w*|metrica\w*|visao)"),
    "problemas_equipe": ("Sobrecarga/falta de pessoal", r"sobrecarga|falta de (gente|pessoal|equipe)|equipe reduzida|turnover|afastament\w+"),
    "gestao_estoque": ("Estoque/logística", r"ruptura|estoque (zerado|baixo)|inventario|divergencia de estoque"),
}

SINAIS_RISCO = {  # sinal -> (regex, peso). Exigem contexto: "cancelamento de nota" é rotina fiscal, não churn.
    "cancelamento/rescisão": (r"(cancel\w+|rescin\w+|encerrar) (o |a |do |da |de )?(contrato|servico|licenca|assinatura|projeto|parceria)|rescis\w+ contratual|(vou|vamos|quero|queremos) cancelar tudo", 0.5),
    "troca de fornecedor": (r"trocar de (sistema|fornecedor|empresa)|migrar para (outr\w+|a concorrencia)|mudar de (sistema|fornecedor)|trocar (o|a) (totvs|sistema)", 0.5),
    "concorrência": (r"concorrente\w*|(para|na|da|a) concorrencia(?! de)", 0.3),
    "insatisfação": (r"insatisf\w+|frustrad\w+|decepcion\w+|pessim\w+|nao (estou|estamos) satisfeit\w+|reclamacao", 0.35),
    "perda de cliente": (r"perder (o|a) (cliente|conta)|risco de perder", 0.5),
    "projeto parado/atrasado": (r"projeto (parado|atrasad\w+)|prazo (estourou|vencid\w+)|fora do prazo|atraso (no|na|do|da) (projeto|entrega|implantacao)", 0.2),
    "problema crítico": (r"producao parada|sistema (fora|parado|fora do ar)|paralisac\w+", 0.3),
    "multa/jurídico": (r"multa (contratual|de rescisao|por rescisao)|notificacao extrajudicial|processo judicial", 0.25),
}
NEGACAO = re.compile(r"nao (quero|vou|pretendo|vamos) (cancelar|trocar|sair|encerrar)|sem (risco|problema) de")

SINAIS_OPORT = {  # "implantação" fica de fora: é projeto em andamento, não venda nova
    "pedido/interesse": (r"temos interesse|tenho interesse|gostar\w+ de (conhecer|contratar|ver)|queremos (contratar|ampliar|adquirir)", 0.4),
    "orçamento/proposta": (r"orcamento|proposta comercial|nova proposta|cotacao", 0.35),
    "expansão/novo módulo": (r"novo modulo|novos modulos|ampliar o (escopo|contrato)|expansao|proxima fase|segunda fase|nova filial|novas unidades", 0.35),
    "treinamento/consultoria": (r"pacote de horas|horas de consultoria|contratar (treinamento|consultoria)|treinamento adicional", 0.25),
}

LEX_POS = r"(obrigad[oa]|otimo|otima|excelente|perfeito|legal|show|parabens|satisfeit\w+|bacana|funcionou|resolveu|resolvido|gostei|tranquilo|feliz)"
LEX_NEG = r"(dificuldade\w*|reclam\w+|atras\w+|preocupad\w+|complicad\w+|travou|parou|falha\w*|insatisf\w+|frustrad\w+|pessim\w+|nao funciona|nao consegu\w+)"

COMPROMISSO = re.compile(
    r"(vou|vamos|fica combinado|ficou de|fico de|preciso|precisamos|vou te|vou enviar|pode enviar|me envia|enviar|"
    r"agendar|marcar|alinhar|verificar|validar|subir|abrir (um )?chamado)[^.?!\n]{8,140}")

CRITICAS = ["producao parada", "risco de perder o cliente", "paralisacao", "sistema fora"]
ALTAS = ["bloqueado", "bloqueada", "bloqueio", "prazo vencido", "fora do prazo", "atraso critico",
         "cliente insatisfeito", "reclamacao grave", "critico"]


# ----------------------------- processamento -----------------------------
def preparar(txt: str) -> str:
    """Remove rótulos de locutor e marcadores de anonimização, preservando as frases."""
    txt = re.sub(r"(?m)^\s*\[LOCUTOR \d+\]:\s*", "", txt)
    txt = re.sub(r"\[LOCUTOR \d+\]:?", " ", txt)
    txt = re.sub(r"\[(PESSOA|EMPRESA|LOCAL|[A-ZÇÃ_ ]{3,20})\]", " ", txt)
    return re.sub(r"[ \t]+", " ", txt).strip()


def normalizar(txt: str) -> str:
    return re.sub(r"\s+", " ", remover_acentos(preparar(txt).lower()))


def _frases(txt: str) -> list[str]:
    return [f.strip() for f in re.split(r"(?<=[.?!])\s+|\n", preparar(txt)) if len(f.strip()) > 25]


def _tokens(txt: str) -> list[str]:
    return [t for t in limpar(preparar(txt)).split()
            if t not in STOPWORDS and t not in STOP_ORAL and len(t) > 2]


# ----------------------------- corpus / termos -----------------------------
@lru_cache(maxsize=1)
def _tfidf():
    reunioes = carregar_reunioes()
    docs = list({r["hash"]: " ".join(_tokens(r["transcricao"])) for r in reunioes}.values())
    vet = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.7, sublinear_tf=True)
    vet.fit(docs)
    return vet


def carregar_reunioes() -> list[dict]:
    if not REAL_JSON.exists():
        raise SystemExit("Importe os dados reais: python -m src.importar_totvs <dataset_limpo.json>")
    return json.load(open(REAL_JSON, encoding="utf-8"))


def termos_chave(txt: str, n: int = 10) -> list[str]:
    vet = _tfidf()
    m = vet.transform([" ".join(_tokens(txt))])
    vocab = vet.get_feature_names_out()
    idx = m.toarray()[0].argsort()[::-1][:n]
    return [str(vocab[i]) for i in idx if m[0, idx[0]] > 0 and m[0, i] > 0]


# ----------------------------- produto -----------------------------
def _contar(rx: str, t: str) -> int:
    return len(re.findall(rf"\b(?:{rx})\b", t))


def pontuar_produtos(t: str) -> dict[str, dict]:
    """Pontua cada produto do catálogo pela densidade de termos (por 1.000 palavras)."""
    n_pal = max(1, len(t.split()))
    out = {}
    for key, prod in CATALOGO_TOTVS.items():
        fortes = {remover_acentos(p.lower()) for p in prod["palavras_fortes"]} | set(EXTRAS_PRODUTO[key])
        hits = {p: _contar(re.escape(p), t) for p in fortes}
        hits = {p: c for p, c in hits.items() if c}
        bruto = sum(1 + math.log(c) for c in hits.values())  # amortece termo repetido
        out[key] = {"pontos": bruto, "densidade": round(1000 * bruto / n_pal, 3),
                    "termos": sorted(hits, key=hits.get, reverse=True)[:6]}
    return out


# ----------------------------- dores / tarefas -----------------------------
def detectar_dores(txt: str) -> list[dict]:
    dores = []
    for cat, (label, rx) in REGRAS_DORES.items():
        achados = [f for f in _frases(txt) if re.search(rx, normalizar(f))]
        for f in achados[:3]:
            dores.append({"categoria": cat, "label": label, "trecho": f[:220]})
        if achados:
            dores[-1]["total"] = len(achados)
    return dores


def extrair_tarefas(txt: str, n: int = 6) -> list[dict]:
    tarefas = []
    for f in _frases(txt):
        m = COMPROMISSO.search(f)
        if m and len(tarefas) < n:
            tarefas.append({"tarefa": m.group(0).strip(), "responsavel": "Não identificado", "prazo": "Não mencionado"})
    return tarefas


# ----------------------------- indicadores -----------------------------
def _score_sinais(t: str, sinais: dict, neg=None) -> tuple[float, list[str]]:
    total, achados = 0.0, []
    for nome, (rx, w) in sinais.items():
        c = _contar(rx, t)
        if c:
            total += w * (1 - 0.5 ** c)  # retorno decrescente: 1 menção ≈ w/2; 3+ ≈ w
            achados.append(f"{nome} (x{c})")
    if neg and neg.search(t):
        total *= 0.5
    return round(100 * min(1.0, total), 1), achados


def sentimento(t: str) -> tuple[str, float]:
    pos, neg = _contar(LEX_POS, t), _contar(LEX_NEG, t)
    score = round(100 * (pos - neg) / (pos + neg + 10), 1)  # +10 amortece textos longos e neutros
    return ("positivo" if score >= 15 else "negativo" if score <= -15 else "neutro"), score


def urgencia(t: str) -> str:
    if any(c in t for c in CRITICAS):
        return "Crítica"
    return "Alta" if any(a in t for a in ALTAS) else "Média" if "urgente" in t or "prioridade" in t else "Baixa"


# ----------------------------- análise -----------------------------
def analisar_reuniao(transcricao: str, segmento: str = "") -> dict:
    t = normalizar(transcricao)
    n_pal = len(t.split())

    pontos = pontuar_produtos(t)
    total = sum(p["pontos"] for p in pontos.values())
    ranking = sorted(pontos, key=lambda k: pontos[k]["pontos"], reverse=True)
    top = ranking[0]
    share = pontos[top]["pontos"] / total if total else 0.0
    sem_sinal = total < 3  # menos de ~3 termos distintos: não afirma produto

    dores = detectar_dores(transcricao)
    tarefas = extrair_tarefas(transcricao)
    urg = urgencia(t)
    tema = ", ".join(pontos[top]["termos"][:3])
    recs = compute_recommendations(dores, tarefas, tema=tema, urgencia=urg, segmento=segmento or "",
                                   full_transcript=transcricao)
    fit = {r["product_key"]: r for r in recs}

    sent, s_sent = sentimento(t)
    s_risco, sin_risco = _score_sinais(t, SINAIS_RISCO, NEGACAO)
    s_op_bruto, sin_op = _score_sinais(t, SINAIS_OPORT)
    fit_top = fit.get(top, {}).get("fit_score", 0.0)
    # oportunidade: interesse declarado + aderência do produto dominante ao que foi discutido
    s_op = round(0.8 * s_op_bruto + 0.2 * 100 * (0 if sem_sinal else share), 1)

    prod = CATALOGO_TOTVS[top]
    if s_risco >= 65:
        prioridade, acao = "Crítica", RECOMENDACOES["retencao_urgente"]
    elif s_op >= 50 and s_risco >= 35:
        prioridade, acao = "Alta", RECOMENDACOES["oportunidade_com_risco"]
    elif s_op >= 50:
        prioridade, acao = "Alta", RECOMENDACOES["oportunidade"]
    elif s_risco >= 35 or sent == "negativo":
        prioridade, acao = "Média", RECOMENDACOES["retencao_preventiva"]
    elif s_op >= 35:
        prioridade, acao = "Média", RECOMENDACOES["oportunidade"]
    else:
        prioridade, acao = "Baixa", RECOMENDACOES["acompanhar"]

    return {
        "produto": None if sem_sinal else prod["product_name"], "produto_key": None if sem_sinal else top,
        "confianca_produto": round(share, 3), "ranking_produtos": [(k, round(pontos[k]["pontos"], 1)) for k in ranking],
        "termos_do_produto": pontos[top]["termos"],
        "sentimento": sent, "score_sentimento": s_sent,
        "risco_churn": s_risco, "nivel_churn": nivel(s_risco), "sinais_churn": sin_risco,
        "oportunidade": s_op, "nivel_oportunidade": nivel(s_op), "sinais_oportunidade": sin_op,
        "urgencia": urg,
        "dores": sorted({d["categoria"] for d in dores}),
        "tarefas_sugeridas": [t_["tarefa"] for t_ in tarefas],
        "principais_termos": termos_chave(transcricao),
        "n_palavras": n_pal, "prioridade": prioridade, "recomendacao": acao,
        "produto_recomendado": None if sem_sinal else {
            "nome": prod["product_name"], "fit_score": fit_top,
            "por_que": fit.get(top, {}).get("why_recommended", f"Termos do módulo em destaque: {', '.join(pontos[top]['termos'][:3])}."),
            "proximo_passo": prod["proximo_passo_implantacao"], "beneficios": prod["beneficios_esperados"][:2]},
    }


def formatar(r: dict) -> str:
    linhas = [
        f"Produto identificado  : {r['produto'] or 'sem sinal suficiente'} (parcela {r['confianca_produto']:.0%}; termos: {', '.join(r['termos_do_produto'][:4])})",
        f"Sentimento            : {r['sentimento']} (score {r['score_sentimento']:+.0f})",
        f"Risco de churn        : {r['risco_churn']:.0f}/100 [{r['nivel_churn']}] {', '.join(r['sinais_churn'])}",
        f"Oportunidade comercial: {r['oportunidade']:.0f}/100 [{r['nivel_oportunidade']}] {', '.join(r['sinais_oportunidade'])}",
        f"Urgência              : {r['urgencia']} · dores: {', '.join(r['dores']) or '—'}",
        f"Principais termos     : {', '.join(r['principais_termos'])}",
        f"Prioridade            : {r['prioridade']}",
        f"Recomendação          : {r['recomendacao']}",
    ]
    if r["produto_recomendado"]:
        linhas.append(f"Próximo passo         : {r['produto_recomendado']['proximo_passo']}")
    return "\n".join(linhas)


if __name__ == "__main__":
    import sys
    txt = open(sys.argv[1], encoding="utf-8").read() if len(sys.argv) > 1 else sys.stdin.read()
    print(formatar(analisar_reuniao(txt)))

In [ ]:
%%writefile src/validate_totvs.py
"""Roda o pipeline nas reuniões reais do Proton Flow e compara com a análise antiga.

Uso: python -m src.validate_totvs
Saídas: reports/analise_reunioes_reais.csv, reports/validacao_reunioes_reais.md, reports/revisao_humana.csv

IMPORTANTE: não existe gabarito humano para estas reuniões. A "análise antiga" (LLM local de 3B parâmetros +
regras) é só um ponto de comparação. Para medir acurácia de verdade, preencha reports/revisao_humana.csv.
"""
import pandas as pd

from . import config
from .totvs_pipeline import analisar_reuniao, carregar_reunioes

PRODUTO_LEGADO = {"fluig": "fluig", "protheus": "protheus", "crm": "crm", "rh": "rh",
                  "analytics": "analytics", "supply": "supply"}


def main():
    reunioes = carregar_reunioes()
    vistos, linhas = set(), []
    for r in reunioes:
        if r["hash"] in vistos:  # mesma transcrição repetida no dataset original
            continue
        vistos.add(r["hash"])
        a = analisar_reuniao(r["transcricao"], "")
        linhas.append({
            "id": r["id"], "data": r["data"], "cliente": r["segmento"], "palavras": a["n_palavras"],
            "produto": a["produto_key"], "parcela_produto": a["confianca_produto"],
            "produto_legado": r["legado"]["produto_top"],
            "sentimento": a["sentimento"], "risco_churn": a["risco_churn"], "nivel_churn": a["nivel_churn"],
            "oportunidade": a["oportunidade"], "urgencia": a["urgencia"], "urgencia_legado": r["legado"]["urgencia"],
            "prioridade": a["prioridade"], "dores": ";".join(a["dores"]),
            "termos": ", ".join(a["principais_termos"][:6]), "sinais_churn": "; ".join(a["sinais_churn"]),
            "sinais_oportunidade": "; ".join(a["sinais_oportunidade"]),
        })
    df = pd.DataFrame(linhas)
    config.REPORTS_DIR.mkdir(exist_ok=True)
    df.to_csv(config.REPORTS_DIR / "analise_reunioes_reais.csv", index=False)

    # gabarito para revisão humana (preencher e reavaliar)
    rev = df[["id", "cliente", "produto", "sentimento", "nivel_churn", "oportunidade"]].copy()
    rev["produto_correto"] = rev["sentimento_correto"] = rev["churn_real (0/1)"] = rev["oportunidade_real (0/1)"] = ""
    rev.to_csv(config.REPORTS_DIR / "revisao_humana.csv", index=False)

    com_leg = df[df.produto_legado.notna() & df.produto.notna()]
    conc = (com_leg.produto == com_leg.produto_legado).mean() if len(com_leg) else float("nan")
    md = [
        "# Análise das reuniões reais (Proton Flow)", "",
        f"- Transcrições únicas analisadas: **{len(df)}** (de {len(reunioes)} registros; duplicatas removidas)",
        f"- Tamanho: mediana de **{int(df.palavras.median())}** palavras (mín. {df.palavras.min()}, máx. {df.palavras.max()})",
        f"- Sem produto identificado (poucos termos de domínio): **{df.produto.isna().sum()}**", "",
        "## Distribuição", "",
        "**Produto dominante**: " + ", ".join(f"{k}: {v}" for k, v in df.produto.value_counts().items()), "",
        "**Sentimento**: " + ", ".join(f"{k}: {v}" for k, v in df.sentimento.value_counts().items()), "",
        "**Risco de churn**: " + ", ".join(f"{k}: {v}" for k, v in df.nivel_churn.value_counts().items()), "",
        "**Prioridade**: " + ", ".join(f"{k}: {v}" for k, v in df.prioridade.value_counts().items()), "",
        "## Comparação com a análise antiga", "",
        f"Concordância no produto principal: **{conc:.0%}** em {len(com_leg)} reuniões com os dois resultados. "
        "A análise antiga indicou `fluig` em 8 das 12 reuniões que ela analisou com transcrição; ela é um ponto de "
        "comparação, não um gabarito, então a divergência não prova erro de nenhum dos dois lados.", "",
        "## Reuniões de maior risco", "",
        df.sort_values("risco_churn", ascending=False).head(5)[["id", "cliente", "produto", "risco_churn", "sinais_churn"]]
        .to_string(index=False), "",
        "## Como validar de verdade", "",
        "Preencha `reports/revisao_humana.csv` (produto correto, sentimento, churn e oportunidade reais) "
        "e compare com as colunas previstas. Conferência manual feita neste trabalho: as duas reuniões de maior risco "
        "(ids 1057714 e 1027294, ambas nível Médio) têm falas reais de saída (\"trocar de fornecedor\", \"decisão é encerrar o projeto\").",
    ]
    (config.REPORTS_DIR / "validacao_reunioes_reais.md").write_text("\n".join(md), encoding="utf-8")
    print("\n".join(md))


if __name__ == "__main__":
    main()

In [ ]:
!python -m src.data_gen
!python -m src.manuais

## 2. Dados e processamento

In [ ]:
import pandas as pd
from src import config
from src.processing import limpar, texto_processado, termos_frequentes

df = pd.read_csv(config.TREINO_CSV)
print(df.shape)
display(df.head(3))
print(df.produto.value_counts(), "\n")
print(df.sentimento.value_counts())

ex = df.transcricao[0]
print("\n--- ORIGINAL ---\n", ex)
print("\n--- PROCESSADO ---\n", texto_processado(ex))

## 3. Modelo: comparação e seleção
Quatro classificadores (Regressão Logística, Naive Bayes, LinearSVC calibrado, Random Forest) sobre TF-IDF,
para cada tarefa (produto, sentimento, churn, oportunidade). Critério: F1 macro em validação cruzada de 5 folds.

In [ ]:
!python -m src.train
comp = pd.read_csv("reports/comparacao_modelos.csv")
display(comp)

In [ ]:
print(open("reports/modelo_selecionado.md", encoding="utf-8").read())

## 4. Indicadores e recomendação para uma nova transcrição

In [ ]:
from src.pipeline import analisar, formatar

transcricao = """
Atendente: Bom dia, em que posso ajudar?
Cliente: Estou muito irritado, o cartão foi cobrado duas vezes na fatura e ninguém resolve. Já liguei três vezes.
Atendente: Peço desculpas, vou abrir um protocolo.
Cliente: Se não estornarem, eu cancelo e vou para a concorrência.
"""
print(formatar(analisar(transcricao)))

### Teste com sua própria transcrição
Edite o texto da caixa e rode a célula.

In [ ]:
#@title Cole a transcrição aqui { display-mode: "form" }
texto = "Atendente: Ol\u00e1!\nCliente: Quero investir mais, o que voc\u00eas t\u00eam com melhor rentabilidade? Tenho interesse em previd\u00eancia." #@param {type:"string"}
print(formatar(analisar(texto.replace("\\n", "\n"))))

## 5. Validação com novas transcrições
- **Sintéticas**: seed diferente do treino (otimista, teste de regressão).
- **Manuais**: escritas à mão; as regras foram calibradas olhando estes casos (não independente).
- **Cegas**: escritas à mão, nunca usadas para ajuste → melhor estimativa de generalização (n pequeno).

In [ ]:
!python -m src.validate
print(open("reports/validacao.md", encoding="utf-8").read())

In [ ]:
d = pd.read_csv("reports/validacao_detalhe_cegas.csv")
display(d[["produto", "pred_produto", "sentimento", "pred_sentimento", "churn", "pred_churn",
           "oportunidade", "pred_oportunidade", "risco_churn", "score_oportunidade", "prioridade"]])

## 6. Painel em lote (todas as transcrições novas)

In [ ]:
novas = pd.read_csv(config.NOVAS_CEGAS_CSV)
res = pd.DataFrame([analisar(t) for t in novas.transcricao])
painel = pd.concat([novas[["id"]], res[["produto", "sentimento", "risco_churn", "oportunidade", "prioridade", "recomendacao"]]], axis=1)
display(painel)
print(painel.prioridade.value_counts())
painel.prioridade.value_counts().plot(kind="bar", title="Transcrições por prioridade");

## 7. Reuniões reais do Proton Flow (transcrições anonimizadas + catálogo TOTVS)
Usa as 30 reuniões reais do projeto `TOTUVsVers-oFinal`. O produto é identificado pelo **catálogo TOTVS** (palavras fortes por
produto) e os indicadores usam sinais léxicos com contexto (ex.: "cancelamento de nota" não conta como churn).
Não há gabarito humano: veja `reports/revisao_humana.csv` para validar de verdade.

In [ ]:
if not os.path.exists("../totvs_proton"):
    !git clone -q https://github.com/Guiroselli/TOTUVsVers-oFinal.git ../totvs_proton
!python -m src.importar_totvs ../totvs_proton/backend/dataset_limpo.json
!python -m src.validate_totvs

In [ ]:
from src.totvs_pipeline import analisar_reuniao, carregar_reunioes, formatar
reunioes = {r["id"]: r for r in carregar_reunioes()}
print(formatar(analisar_reuniao(reunioes["1057714"]["transcricao"])))

In [ ]:
real = pd.read_csv("reports/analise_reunioes_reais.csv")
display(real[["id", "cliente", "produto", "sentimento", "risco_churn", "oportunidade", "prioridade", "termos"]])
real.produto.value_counts().plot(kind="bar", title="Produto TOTVS dominante por reunião");

## 8. Recomendações de negócio, limitações e próximos passos
Veja o `README.md` do repositório para o detalhamento completo:
https://github.com/Guiroselli/sprint4Nemec#readme

**Resumo:** retenção em até 24h para churn ≥ 65; não vender para cliente insatisfeito (resolver primeiro);
fila de leads para oportunidade ≥ 50; monitorar termos de churn.
**Limitações:** dados sintéticos, validação cega com apenas 8 casos, sentimento é a tarefa mais frágil (75% no cego).
**Próximos passos:** treinar com transcrições reais, ampliar o conjunto cego, rotular churn com desfecho real,
testar embeddings (ex.: BERTimbau) e otimizar limiares por custo de negócio.